# Retail Rocket Fake Discount Detector: Feature Engineering

The purpose of this notebook is to create a leakage-safe historical feature dataset using our newly validated Retail Rocket price-history data.

**CRITICAL DATA SCIENCE REQUIREMENT:** AVOID DATA LEAKAGE.
For each price observation at date D, historical features must be calculated using ONLY price observations strictly BEFORE date D (i.e. `date < D`).


## STEP 1 — LOAD AND VALIDATE

**Why this matters:** We ensure the integrity of the data we're working with before applying transformations.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
import os
DATA_DIR = 'data' if os.path.exists('data') else ('../data' if os.path.exists('../data') else 'data')
original_df = pd.read_csv(os.path.join(DATA_DIR, 'retailrocket_price_history.csv')

# Convert date to datetime
original_df['date'] = pd.to_datetime(original_df['date'])

# Sort data by product_id and date
original_df = original_df.sort_values(by=['product_id', 'date']).reset_index(drop=True)

print(f"Shape: {original_df.shape}")
print(f"Columns: {list(original_df.columns)}")
print("\nData types:")
print(original_df.dtypes)
print("\nMissing values:")
print(original_df.isna().sum())
print(f"\nDuplicate rows: {original_df.duplicated().sum()}")
print(f"Date range: {original_df['date'].min().date()} to {original_df['date'].max().date()}")
print(f"Unique products: {original_df['product_id'].nunique():,}")
print(f"Unique dates: {original_df['date'].nunique():,}")

original_df.head()


Shape: (1790516, 3)
Columns: ['product_id', 'date', 'price']

Data types:
product_id             int64
date          datetime64[us]
price                float64
dtype: object

Missing values:
product_id    0
date          0
price         0
dtype: int64

Duplicate rows: 0
Date range: 2015-05-10 to 2015-09-13
Unique products: 417,053
Unique dates: 18


,product_id,date,price
0,0,2015-05-10,91200.0
1,1,2015-05-10,5760.0
2,1,2015-05-17,5760.0
3,1,2015-05-24,5760.0
4,1,2015-05-31,5760.0


## STEP 2 — HANDLE SAME-DAY OBSERVATIONS

**Why this matters:** A product may have multiple price changes on the same day. To avoid data leakage (where the evening price acts as 'history' for the morning price, or vice-versa), we aggregate multiple daily observations into a single daily representative price (the median). This ensures a stable daily timeline.


In [5]:
same_day_counts = original_df.groupby(['product_id', 'date']).size()
duplicates = same_day_counts[same_day_counts > 1]
print(f"Products with same-day multiple observations: {duplicates.reset_index()['product_id'].nunique():,}")

# Create internal daily aggregation
product_daily = original_df.groupby(['product_id', 'date'])['price'].median().reset_index()
product_daily = product_daily.sort_values(['product_id', 'date'])


Products with same-day multiple observations: 0


## STEP 3 — CREATE STRICT HISTORICAL FEATURES

**Why this matters:** We use our stable daily timeline to calculate rolling historical statistics that **strictly** look at dates prior to the current observation.


In [6]:
import time
print("--- Starting STEP 3 ---")
step3_start_time = time.time()

# 1. Sort product_daily to ensure correct chronological order for cumulative operations
product_daily = product_daily.sort_values(['product_id', 'date'])

# 2. Shift price within each product group
shifted_price = product_daily.groupby('product_id')['price'].shift(1)
product_daily['product_previous_price'] = shifted_price

# 3. Calculate previous count using cumcount on the sorted dataframe
# cumcount starts at 0, which matches the count of *strictly previous* observations
product_daily['product_previous_count'] = product_daily.groupby('product_id').cumcount()
# For the first observation (where previous count should be 0), we want it to be 0 instead of NaN.
# Wait, expanding().count() returns 1 when there's 1 non-NaN element.
# The original code:
# product_daily['product_previous_count'] = grouped_product.transform(lambda x: x.shift(1).expanding().count())
# This would return 0.0 or NaN for the first element, then 1.0, 2.0...
# product_daily.groupby('product_id').cumcount() gives 0, 1, 2... exactly. 

# 4. Calculate historical min and max using cumulative operations on the shifted price
product_daily['product_historical_min'] = shifted_price.groupby(product_daily['product_id']).cummin()
product_daily['product_historical_max'] = shifted_price.groupby(product_daily['product_id']).cummax()

# 5. Calculate historical mean and std using cumulative sums
cum_sum = shifted_price.groupby(product_daily['product_id']).cumsum()
product_daily['product_historical_mean'] = cum_sum / product_daily['product_previous_count']

# Calculate cumulative sum of squares for standard deviation
cum_sum_sq = (shifted_price ** 2).groupby(product_daily['product_id']).cumsum()
# Variance = (sum(x^2) - (sum(x))^2 / n) / (n - 1)
variance = (cum_sum_sq - (cum_sum ** 2) / product_daily['product_previous_count']) / (product_daily['product_previous_count'] - 1)
# Prevent negative variance due to floating point precision
product_daily['product_historical_std'] = np.sqrt(np.maximum(variance, 0))

# 6. Calculate historical median
# Since expanding().median() on groupby is very slow, and max dates per product is small (18),
# we can horizontally concat shifted prices and use vectorized median(axis=1) which is much faster.
max_obs = product_daily.groupby('product_id').size().max()
shifted_cols = [product_daily.groupby('product_id')['price'].shift(i) for i in range(1, max_obs)]
product_daily['product_historical_median'] = pd.concat(shifted_cols, axis=1).median(axis=1)

# Drop the aggregated price so we don't conflict with the original price column
product_daily = product_daily.drop(columns=['price'])

# Merge back into the original observation-level dataframe
df = original_df.merge(product_daily, on=['product_id', 'date'], how='left')

step3_end_time = time.time()
print(f"STEP 3 completed in {step3_end_time - step3_start_time:.2f} seconds.")

# Performance diagnostics
print("Rows in product_daily:", len(product_daily))
print("Rows in original:", len(original_df))
print("Rows after merge:", len(df))

assert len(original_df) == len(df)

--- Starting STEP 3 ---
STEP 3 completed in 8.53 seconds.
Rows in product_daily: 1790516
Rows in original: 1790516
Rows after merge: 1790516


## STEP 4 — PRICE CHANGE FEATURES

**Why this matters:** We calculate immediate jumps and relative discounts from long-term baselines. This defines how 'deep' a discount claims to be.


In [7]:
# Immediate price change
df['price_change'] = df['price'] - df['product_previous_price']

df['price_change_pct'] = np.where(
    df['product_previous_price'].isna() | (df['product_previous_price'] == 0), 
    np.nan, 
    (df['price_change'] / df['product_previous_price']) * 100
)

# Compare current price vs product median
df['price_vs_product_median_pct'] = np.where(
    df['product_historical_median'].isna() | (df['product_historical_median'] == 0),
    np.nan,
    ((df['price'] - df['product_historical_median']) / df['product_historical_median']) * 100
)

# Discount from product median
df['discount_from_product_median_pct'] = np.where(
    df['product_historical_median'].isna() | (df['product_historical_median'] == 0),
    np.nan,
    ((df['product_historical_median'] - df['price']) / df['product_historical_median']) * 100
)

# Discount from product max
df['discount_from_product_max_pct'] = np.where(
    df['product_historical_max'].isna() | (df['product_historical_max'] == 0),
    np.nan,
    ((df['product_historical_max'] - df['price']) / df['product_historical_max']) * 100
)


## STEP 5 — HISTORY SUFFICIENCY

**Why this matters:** We need to flag observations that lack enough context (e.g., fewer than 3 previous dates) so our models can treat them cautiously without deleting them entirely.


In [8]:
df['product_history_sufficient'] = df['product_previous_count'] >= 3

# Diagnostic stats
print(f"Rows with previous_count >= 1: {(df['product_previous_count'] >= 1).sum():,}")
print(f"Rows with previous_count >= 3: {(df['product_previous_count'] >= 3).sum():,}")
print(f"Rows with previous_count >= 5: {(df['product_previous_count'] >= 5).sum():,}")
print(f"Rows with previous_count >= 10: {(df['product_previous_count'] >= 10).sum():,}")


Rows with previous_count >= 1: 1,373,463
Rows with previous_count >= 3: 1,203,244
Rows with previous_count >= 5: 1,034,871
Rows with previous_count >= 10: 621,935


## STEP 6 — PRICE VOLATILITY

**Why this matters:** A large discount on a highly volatile product is less suspicious than a large discount on a rock-solid, stable product.


In [9]:
# Volatility is strictly only calculated if we have sufficient history to avoid noise
df['product_price_volatility'] = np.where(df['product_history_sufficient'], df['product_historical_std'], np.nan)

# Coefficient of Variation (CV = std / mean)
df['historical_cv'] = np.where(
    df['product_history_sufficient'] & df['product_historical_mean'].notna() & (df['product_historical_mean'] != 0),
    df['product_historical_std'] / df['product_historical_mean'],
    np.nan
)


## STEP 7 — PRESERVE ORIGINAL ROWS

**Why this matters:** We must assert that our methodology didn't accidentally drop any rows or duplicate them through bad merges.


In [10]:
assert len(original_df) == len(df), f"Row count mismatch! Original: {len(original_df)}, Output: {len(df)}"
print(f"Row count preservation successful! Output has exactly {len(df):,} rows.")


Row count preservation successful! Output has exactly 1,790,516 rows.


## STEP 8 — LEAKAGE VALIDATION

**Why this matters:** We must manually prove to stakeholders that the logic works and that same-day data points share an identical baseline (without leaking).


In [11]:
# Find a product with multiple observations on the same date
dup_mask = df.duplicated(subset=['product_id', 'date'], keep=False)
if dup_mask.sum() > 0:
    sample_product = df[dup_mask].iloc[0]['product_id']
else:
    # If no same-day duplicates, just pick one with a long history
    sample_product = df.nlargest(1, 'product_previous_count').iloc[0]['product_id']

inspection_cols = [
    'date', 'price', 'product_previous_price', 'product_historical_median', 
    'product_historical_max', 'product_previous_count'
]
leakage_check_df = df[df['product_id'] == sample_product][inspection_cols].head(15)

print(f"Leakage Validation for Product: {sample_product}")
display(leakage_check_df)


Leakage Validation for Product: 1


,date,price,product_previous_price,product_historical_median,product_historical_max,product_previous_count
1,2015-05-10,5760.0,NaN,NaN,NaN,0
2,2015-05-17,5760.0,5760.0,5760.0,5760.0,1
3,2015-05-24,5760.0,5760.0,5760.0,5760.0,2
4,2015-05-31,5760.0,5760.0,5760.0,5760.0,3
5,2015-06-07,5760.0,5760.0,5760.0,5760.0,4
6,2015-06-14,6240.0,5760.0,5760.0,5760.0,5
7,2015-06-28,6000.0,6240.0,5760.0,6240.0,6
8,2015-07-05,6000.0,6000.0,5760.0,6240.0,7
9,2015-07-12,6120.0,6000.0,5760.0,6240.0,8
10,2015-07-19,6120.0,6120.0,5760.0,6240.0,9


## STEP 9 — FEATURE QUALITY REPORT

In [12]:
print("=== Feature Quality Report ===")
print(f"Total rows: {len(df):,}")
print(f"Unique products: {df['product_id'].nunique():,}")
print(f"Unique dates: {df['date'].nunique():,}\n")

print("Number of rows with:")
print(f"- previous_count >= 1: {(df['product_previous_count'] >= 1).sum():,}")
print(f"- previous_count >= 3: {(df['product_previous_count'] >= 3).sum():,}")
print(f"- previous_count >= 5: {(df['product_previous_count'] >= 5).sum():,}")
print(f"- previous_count >= 10: {(df['product_previous_count'] >= 10).sum():,}\n")

# To calculate unique products matching these thresholds, we check the maximum previous_count per product
max_counts = df.groupby('product_id')['product_previous_count'].max()

print("Number of products with:")
print(f"- >=3 historical dates: {(max_counts >= 3).sum():,}")
print(f"- >=5 historical dates: {(max_counts >= 5).sum():,}")
print(f"- >=10 historical dates: {(max_counts >= 10).sum():,}\n")

print("Number of rows with:")
price_changes = df['price_change'].notna() & (df['price_change'] != 0)
positive_changes = df['price_change'] > 0
negative_changes = df['price_change'] < 0

print(f"- price changes: {price_changes.sum():,}")
print(f"- positive price changes: {positive_changes.sum():,}")
print(f"- negative price changes: {negative_changes.sum():,}")


=== Feature Quality Report ===
Total rows: 1,790,516
Unique products: 417,053
Unique dates: 18

Number of rows with:
- previous_count >= 1: 1,373,463
- previous_count >= 3: 1,203,244
- previous_count >= 5: 1,034,871
- previous_count >= 10: 621,935

Number of products with:
- >=3 historical dates: 84,420
- >=5 historical dates: 83,633
- >=10 historical dates: 80,850

Number of rows with:
- price changes: 381,093
- positive price changes: 228,756
- negative price changes: 152,337


## STEP 10 — SAVE

In [13]:
output_path = '../data/retailrocket_fake_discount_features.csv'
df.to_csv(output_path, index=False)

print(f"Saved strictly leakage-safe features to: {output_path}")


Saved strictly leakage-safe features to: ../data/retailrocket_fake_discount_features.csv


# Summary of Feature Engineering

1. **Features Created**: Strict historical baselines (min, max, median, mean, std) at the product level, alongside immediate price changes, percentage discounts relative to historical medians/maximums, and volatility metrics (std & CV).
2. **Leakage Prevention**: To guarantee no future or current-day information leaked into the historical baselines, we aggregated same-day scrapes into a daily median timeline and then applied a strict `shift(1)` operation. This mathematically forces an observation at date `D` to rely *only* on prices strictly before `D`. Same-day observations cleanly inherit the exact same un-leaked historical baseline.
3. **Data Sufficiency**: A massive volume of rows and products maintain robust historical records (`previous_count >= 3`), establishing a solid foundation for detecting anomalous discounts. (Exact numbers printed in the Quality Report).
4. **Output Location**: Saved securely to `data/retailrocket_fake_discount_features.csv`, preserving all 1,790,516 original rows.
